# 05 – Multi-Run Comparison

Side-by-side comparison of all OIFS SST-sensitivity experiments for Hurricane Kirk:
- **baserun**: Control (observed SST)
- **+3K**: SST +3 K warming
- **+6K**: SST +6 K warming

Panels:
1. Storm track map (all runs + IBTrACS)
2. MSLP minimum time series
3. Cyclone Phase Space overlay
4. R17/R34 wind radius time series

In [1]:
import sys, os
sys.path.insert(0, '../scripts')
sys.path.insert(0, '../ribberink_code')

import numpy as np
import xarray as xr
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.cm as cm

try:
    import cartopy.crs as ccrs
    import cartopy.feature as cfeature
    HAS_CARTOPY = True
except ImportError:
    HAS_CARTOPY = False
    print('cartopy not installed – map plot will use plain axes')

from oifs_adapter import RUNS, OIFSRun
from run_track import load_kirk_best_track, run_name_to_safe

## 1. Load pre-computed tracks and CPS data

In [2]:
# Prefer explicit run order so all known experiments are included, especially -3K.
run_order = ['-3K', 'baserun', '+3K', '+6K']

tracks = {}
for run_name in run_order:
    safe_name = run_name_to_safe(run_name)
    path = f'../plots/tracks/track_{safe_name}.nc'
    if os.path.exists(path):
        tracks[run_name] = xr.open_dataset(path)
        print(f'Track loaded: {run_name} ({len(tracks[run_name].time)} steps)')
    else:
        print(f'Track missing: {run_name} ({path})')

cps = {}
for run_name in run_order:
    safe_name = run_name_to_safe(run_name)
    path = f'../plots/cps/cps_{safe_name}.nc'
    if os.path.exists(path):
        cps[run_name] = xr.open_dataset(path)
        print(f'CPS loaded: {run_name}')
    else:
        print(f'CPS missing: {run_name} ({path})')

Track loaded: -3K (49 steps)
Track loaded: baserun (49 steps)
Track loaded: +3K (49 steps)
Track loaded: +6K (49 steps)
CPS missing: -3K (../plots/cps/cps_m3K.nc)
CPS missing: baserun (../plots/cps/cps_baserun.nc)
CPS missing: +3K (../plots/cps/cps_p3K.nc)
CPS missing: +6K (../plots/cps/cps_p6K.nc)


In [3]:
# Load IBTrACS best track using robust path fallbacks from scripts/run_track.py
obs_track = None
try:
    obs_track = load_kirk_best_track().to_dataframe().reset_index()
    print(f'IBTrACS Kirk loaded: {len(obs_track)} records')
except Exception as exc:
    print(f'IBTrACS load failed: {exc}')
    obs_track = None

Best track loaded from IBTrACS: 83 timesteps, 2024-09-29 18:00:00 -> 2024-10-10 00:00:00
IBTrACS Kirk loaded: 83 records


## 2. Storm Track Map

In [4]:
# Match run colors with scripts/plot_tracks_comparison.py
colors = {'-3K': 'tab:blue', 'baserun': 'tab:green', '+3K': 'tab:orange', '+6K': 'tab:red'}

if HAS_CARTOPY:
    fig = plt.figure(figsize=(12, 7))
    ax = fig.add_subplot(1, 1, 1, projection=ccrs.PlateCarree())
    ax.add_feature(cfeature.LAND, facecolor='#d9d9d9')
    ax.add_feature(cfeature.COASTLINE, linewidth=0.5)
    ax.gridlines(draw_labels=True, linewidth=0.3)
    ax.set_extent([-90, -5, 5, 60], crs=ccrs.PlateCarree())
    transform = ccrs.PlateCarree()
else:
    fig, ax = plt.subplots(figsize=(12, 7))
    transform = None

for run_name, track in tracks.items():
    kw = dict(color=colors.get(run_name, 'k'), lw=2, label=run_name)
    if transform:
        ax.plot(track.lon, track.lat, transform=transform, **kw)
    else:
        ax.plot(track.lon, track.lat, **kw)

if obs_track is not None:
    kw = dict(color='k', lw=2, ls='--', label='IBTrACS')
    if transform:
        ax.plot(obs_track['lon'], obs_track['lat'], transform=transform, **kw)
    else:
        ax.plot(obs_track['lon'], obs_track['lat'], **kw)

ax.set_title('Hurricane Kirk 2024 – Storm Tracks (OIFS + IBTrACS)')
ax.legend(loc='upper left')
plt.tight_layout()
plt.savefig('../plots/kirk_track_comparison.png', dpi=150)
plt.show()
print('Saved: ../plots/kirk_track_comparison.png')

Saved: ../plots/kirk_track_comparison.png


## 3. MSLP Time Series

In [5]:
fig, ax = plt.subplots(figsize=(11, 5))

# Load ET-transition start times from timing analysis summary
ett_summary_path = '../plots/tracks/ett_start_summary.csv'
ett_start_times = {}
if os.path.exists(ett_summary_path):
    import pandas as pd
    ett_df = pd.read_csv(ett_summary_path)
    for _, row in ett_df.iterrows():
        run_name = str(row.get('run', ''))
        et_start = row.get('et_start_time_utc', '')
        if run_name and isinstance(et_start, str) and et_start.strip():
            ett_start_times[run_name] = np.datetime64(et_start)

for run_name, track in tracks.items():
    t_vals = np.array(track.time.values)
    msl_vals = np.array(track.msl.values) / 100.0
    c = colors.get(run_name, 'k')

    ax.plot(t_vals, msl_vals, '.-', color=c, label=run_name)

    # Larger marker for ETT start time (same run color)
    et_t = ett_start_times.get(run_name)
    if et_t is not None and len(t_vals) > 0:
        idx_et = int(np.abs(t_vals.astype('datetime64[ns]') - et_t.astype('datetime64[ns]')).argmin())
        ax.plot(t_vals[idx_et], msl_vals[idx_et], 'o', ms=12, color=c,
                markeredgecolor='k', markeredgewidth=0.8, zorder=6)

if obs_track is not None and 'pres' in obs_track.columns:
    ax.plot(obs_track['datetime'], obs_track['pres'].astype(float),
            'k--', label='IBTrACS (obs)')

ax.set_xlabel('Date')
ax.set_ylabel('MSLP (hPa)')
ax.set_title('Hurricane Kirk 2024 – Minimum Sea-Level Pressure')
ax.legend()
ax.grid(True, alpha=0.3)
plt.setp(ax.xaxis.get_majorticklabels(), rotation=30)
plt.tight_layout()
plt.savefig('../plots/kirk_mslp_comparison.png', dpi=150)
plt.show()
print('Saved: ../plots/kirk_mslp_comparison.png')

Saved: ../plots/kirk_mslp_comparison.png


## 4. Cyclone Phase Space Overlay

In [19]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
panel_titles = ['Upper CPS (300-600 hPa)', 'Lower CPS (600-900 hPa)']

for ax, (key_sym, key_th), title in zip(
    axes,
    [('B', 'VT_upper'), ('B', 'VT_lower')],
    panel_titles
):
    for run_name, cps_ds in cps.items():
        if key_sym in cps_ds and key_th in cps_ds:
            B = cps_ds[key_sym].values
            VT = cps_ds[key_th].values
            ax.plot(B, VT, '.-', color=colors.get(run_name, 'k'), label=run_name)
            ax.plot(B[0], VT[0], 'o', ms=8, color=colors.get(run_name, 'k'))
            ax.plot(B[-1], VT[-1], 's', ms=8, color=colors.get(run_name, 'k'))

    ax.axhline(0, color='k', lw=0.8)
    ax.axvline(0, color='k', lw=0.8)
    ax.set_xlabel('B (Thermal asymmetry, m)')
    ax.set_ylabel('$-V_T$ (m)')
    ax.set_title(title)
    ax.text(+20, +50, 'Warm\nSymmetric', ha='center', va='center', fontsize=9, color='#555')
    ax.text(-20, +50, 'Warm\nAsymmetric', ha='center', va='center', fontsize=9, color='#555')
    ax.text(+20, -50, 'Cold\nSymmetric', ha='center', va='center', fontsize=9, color='#555')
    ax.text(-20, -50, 'Cold\nAsymmetric', ha='center', va='center', fontsize=9, color='#555')

axes[0].legend(loc='upper right')
fig.suptitle('Hurricane Kirk 2024 – Cyclone Phase Space (Hart 2003)', fontsize=13)
plt.tight_layout()
plt.savefig('../plots/kirk_cps_overlay.png', dpi=150)
plt.show()
print('Saved: ../plots/kirk_cps_overlay.png')

/tmp/ipykernel_1663821/4261566148.py:27: UserWarning: No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.
  axes[0].legend(loc='upper right')
/tmp/ipykernel_1663821/4261566148.py:29: UserWarning: Tight layout not applied. The left and right margins cannot be made large enough to accommodate all Axes decorations.
  plt.tight_layout()


Saved: ../plots/kirk_cps_overlay.png


## 5. Summary statistics

In [16]:
import pandas as pd

rows = []
for run_name, track in tracks.items():
    mslp_min = float(track.msl.min()) / 100  # hPa
    lat_at_min = float(track.lat.isel(time=int(track.msl.argmin())))
    rows.append({
        'Run': run_name,
        'Min MSLP (hPa)': f'{mslp_min:.1f}',
        'Lat at min': f'{lat_at_min:.1f}°N',
        'Track length (steps)': len(track.time)
    })

df = pd.DataFrame(rows).set_index('Run')
print(df.to_string())
df.to_csv('../plots/kirk_summary_stats.csv')
print('Saved: ../plots/kirk_summary_stats.csv')

        Min MSLP (hPa) Lat at min  Track length (steps)
Run                                                    
-3K              972.9     19.8°N                    49
baserun          964.2     41.5°N                    49
+3K              933.6     39.8°N                    49
+6K              893.0     36.4°N                    49
Saved: ../plots/kirk_summary_stats.csv


/users/jfkarhu/Numlab/hurricane_kirk/.venv/lib64/python3.11/site-packages/xarray/core/dataarray.py:6285: FutureWarning: Behaviour of argmin/argmax with neither dim nor axis argument will change to return a dict of indices of each dimension. To get a single, flat index, please use np.argmin(da.data) or np.argmax(da.data) instead of da.argmin() or da.argmax().
  result = self.variable.argmin(dim, axis, keep_attrs, skipna)
/users/jfkarhu/Numlab/hurricane_kirk/.venv/lib64/python3.11/site-packages/xarray/core/dataarray.py:6285: FutureWarning: Behaviour of argmin/argmax with neither dim nor axis argument will change to return a dict of indices of each dimension. To get a single, flat index, please use np.argmin(da.data) or np.argmax(da.data) instead of da.argmin() or da.argmax().
  result = self.variable.argmin(dim, axis, keep_attrs, skipna)
/users/jfkarhu/Numlab/hurricane_kirk/.venv/lib64/python3.11/site-packages/xarray/core/dataarray.py:6285: FutureWarning: Behaviour of argmin/argmax with 